# Bose2 — real crochet footage + recorded voice (free Colab)

Use this notebook from a phone or computer. Upload your own recorded narration and crochet hand-footage that you own or have permission to use. The output is a 1080p MP4 with Arabic captions. No face/avatar, paid API, GPU, billing, or automatic publishing is used.

**Important:** use one narration audio file and one or more video clips. The audio can be MP3, M4A, WAV, or AAC and will be converted to MP3 locally in this free runtime. Video clips can be MP4, MOV, MKV, or WEBM. Keep the page open while rendering; free Colab runtimes may disconnect.

In [ ]:
# Set up a fresh temporary runtime and get the real-footage provider from this PR branch.
!git clone --depth 1 --branch feature/real-hands-footage-mode https://github.com/bosesamer-code/Bose2.git /content/Bose2
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-core >/dev/null
import os, sys, subprocess
from pathlib import Path
sys.path.insert(0, '/content/Bose2')
from src.media_providers import MediaRequest
from src.real_footage_providers import RecordedAudioProvider, RealFootageVideoProvider
print('Setup complete. No paid service or GPU required.')

In [ ]:
from google.colab import files
uploaded = files.upload()  # Choose one narration file and your crochet video clips.
input_dir = Path('/content/bose2_inputs')
input_dir.mkdir(parents=True, exist_ok=True)
saved = []
for name, data in uploaded.items():
    path = input_dir / Path(name).name
    path.write_bytes(data)
    saved.append(path)
audio_candidates = [p for p in saved if p.suffix.lower() in {'.mp3', '.m4a', '.wav', '.aac'}]
video_candidates = [p for p in saved if p.suffix.lower() in {'.mp4', '.mov', '.mkv', '.webm'}]
if len(audio_candidates) != 1:
    raise ValueError(f'Upload exactly one narration audio file; found {len(audio_candidates)}.')
if not video_candidates:
    raise ValueError('Upload at least one real crochet video clip (MP4/MOV/MKV/WEBM).')
audio_mp3 = input_dir / 'recorded_narration.mp3'
subprocess.run(['ffmpeg', '-y', '-i', str(audio_candidates[0]), '-vn', '-codec:a', 'libmp3lame', '-q:a', '3', str(audio_mp3)], check=True, capture_output=True)
print('Narration:', audio_candidates[0].name)
print('Footage clips:', len(video_candidates))
print('Review the uploaded files before continuing. Only use media you have rights to use.')

## Edit captions before rendering

Replace the example caption text below with your lesson's Arabic steps. Set approximate scene lengths; the renderer extends timing when the narration is longer. Clips are reused in order when there are fewer clips than scenes.

In [ ]:
scene_plan = [
    {'scene': 1, 'purpose': 'hook', 'duration_seconds': 15, 'text': 'اكتبي هنا مقدمة الدرس'},
    {'scene': 2, 'purpose': 'lesson', 'duration_seconds': 25, 'text': 'اكتبي هنا الخطوة الأولى'},
    {'scene': 3, 'purpose': 'application', 'duration_seconds': 30, 'text': 'اكتبي هنا الخطوة الثانية'},
    {'scene': 4, 'purpose': 'cta', 'duration_seconds': 20, 'text': 'اكتبي هنا الخلاصة'},
]
job_dir = Path('/content/bose2_real_footage_job')
audio_result = RecordedAudioProvider().generate(MediaRequest('crochet-lesson', job_dir / 'audio', {'recorded_audio_path': str(audio_mp3)}))
if audio_result.status != 'completed':
    raise RuntimeError(audio_result.error or 'Recorded narration setup failed.')
payload = {
    'recorded_audio_path': str(audio_mp3),
    'audio_path': audio_result.artifacts[0],
    'real_footage_paths': [str(p) for p in video_candidates],
    'scene_plan': scene_plan,
]
result = RealFootageVideoProvider().generate(MediaRequest('crochet-lesson', job_dir / 'video', payload))
print('Status:', result.status)
if result.status != 'completed':
    raise RuntimeError(result.error or 'Video render failed.')
video_path = Path(result.artifacts[0])
print('Video created:', video_path, 'size:', video_path.stat().st_size, 'bytes')

In [ ]:
# Download the MP4 to your phone/computer. Watch it all the way through before sharing.
from google.colab import files
files.download(str(video_path))